# UniqueShip Fold 1 — unsupervised STFT clustering by vessel distance

**Goal:** Does comparing vessels at similar hydrophone distances reveal clearer vessel-type clusters?

This notebook **reuses the original 128-bin log-STFT mean/std feature cache (1,206 dimensions)** from our existing five-class supervised experiments. **No WAV extraction, model retraining, or test-set evaluation.**

Run the cells from top to bottom in VS Code. It produces:

- **Two plots of the exact same points:** coloured by K-Means cluster and actual vessel class.
- A **cluster-versus-class heatmap**, plus ARI/NMI scores.
- A **distance-window experiment**, including a size/class-matched control.
- An optional interactive **distance slider**.

**Dataset:** five classes = background, cargo, passenger ship, tanker, tug. Distance-specific experiments exclude background, leaving **four vessel classes and K=4**. The original author Fold 1 **training** features fit the scaling/PCA representation; the original **validation** recordings are used for exploratory clustering and class-label evaluation. No labels are supplied to PCA or K-Means.

## 1 · Imports and paths

Use the Python interpreter of your `UnderwaterVesselDeployment/.venv` environment. If needed, install `numpy pandas matplotlib scikit-learn ipykernel` there. The optional slider also needs `ipywidgets` (`%pip install ipywidgets`).

In [ ]:
from pathlib import Path
import hashlib
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

SEED = 42
FOLD = 1
PCA_COMPONENTS = 32  # number used for clustering; plots use the first 2
PCA_FIT_SAMPLES = 12_000  # random training examples to fit scaler/PCA efficiently
PLOT_MAX_POINTS = 6_000  # display-only limit; ARI/NMI use every selected clip
DISTANCE_UNITS = "auto"  # "auto", "km", or "m"; checked below
WINDOW_MIN_M, WINDOW_MAX_M = 400, 700

DATA = Path.home() / "Local Documents/Datasets/UniqueShip/5class_balanced"
CSV_DIR = DATA / "audio/main5_balanced" / f"fold_{FOLD}"
CACHE_DIR = DATA / "feature_cache/stft_axis_stats_v1"  # ORIGINAL log-frequency STFT cache

# Locate the repository from VS Code's current directory (root or a subfolder).
TRAINING_SCRIPT = Path("models/training/uniqueship/train_stft_classical.py")
SEARCH_ROOTS = [Path.cwd(), Path.home()/"PycharmProjects/UnderwaterVesselDeployment"]
REPO = next((p for r in SEARCH_ROOTS for p in [r, *r.parents]
             if (p / TRAINING_SCRIPT).is_file()), Path.cwd())
OUT = REPO / "benchmarks/results/uniqueship_stft/fold1_unsupervised_clustering"
OUT.mkdir(parents=True, exist_ok=True)

for path in (CSV_DIR / "metadata_5s_train.csv", CSV_DIR / "metadata_5s_validation.csv", CACHE_DIR):
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}\nCheck the DATA / CSV_DIR / CACHE_DIR settings above.")
print("Original STFT cache:", CACHE_DIR)
print("Results and figures:", OUT)

## 2 · Load the existing feature matrices and matching metadata

**Important:** the cache filenames contain a hash of the ordered audio path/label metadata. We verify that hash, so we don't silently pair the wrong feature rows with the labels. Files from the 64-band/power-law experiments are **not** used.

In [ ]:
train_meta = pd.read_csv(CSV_DIR / "metadata_5s_train.csv", dtype=str, low_memory=False).reset_index(drop=True)
val_meta = pd.read_csv(CSV_DIR / "metadata_5s_validation.csv", dtype=str, low_memory=False).reset_index(drop=True)
assert {"label", "path", "mean_distance"}.issubset(train_meta.columns)
assert {"label", "path", "mean_distance"}.issubset(val_meta.columns)

# Hash used by the original train_stft_classical_v3.py.
FRONTEND_VERSION = "uniqueship_shipsear_stft_axis_stats_v1"
def expected_cache_path(metadata, split):
    h = hashlib.sha256()
    h.update(FRONTEND_VERSION.encode())
    for path, label in metadata[["path", "label"]].itertuples(index=False, name=None):
        h.update(str(path).encode()); h.update(b"\x00")
        h.update(str(label).encode()); h.update(b"\n")
    return CACHE_DIR / f"fold{FOLD}_{split}_{h.hexdigest()[:14]}_X.npy"

# Only set overrides if the original script used a different frontend-version string.
# Never point these to the new uniform64 or powerlaw64 feature caches.
TRAIN_CACHE_OVERRIDE = None
VAL_CACHE_OVERRIDE = None

train_file = Path(TRAIN_CACHE_OVERRIDE).expanduser() if TRAIN_CACHE_OVERRIDE else expected_cache_path(train_meta, "train")
val_file = Path(VAL_CACHE_OVERRIDE).expanduser() if VAL_CACHE_OVERRIDE else expected_cache_path(val_meta, "validation")
for split, f in (("train", train_file), ("validation", val_file)):
    if not f.is_file():
        other = list(CACHE_DIR.glob(f"fold{FOLD}_{split}_*_X.npy"))
        raise FileNotFoundError(
            f"Could not find the metadata-matched {split} cache:\n  {f}\n"
            f"Available candidates: {[p.name for p in other]}\n"
            "If you know the original 1,206-D cache file corresponds to these exact CSV rows, "
            "set the corresponding *_CACHE_OVERRIDE above. Otherwise regenerate the original cache."
        )

X_train = np.load(train_file, mmap_mode="r")
X_val = np.load(val_file, mmap_mode="r")
assert X_train.shape == (len(train_meta), 1206), (X_train.shape, len(train_meta))
assert X_val.shape == (len(val_meta), 1206), (X_val.shape, len(val_meta))

labels_val = val_meta["label"].astype(str).str.strip().str.lower().to_numpy()
class_names = sorted(np.unique(labels_val).tolist())
ship_names = [name for name in class_names if name != "background"]
assert len(class_names) == 5 and len(ship_names) == 4, class_names

print("Training features:", X_train.shape, train_file.name)
print("Validation features:", X_val.shape, val_file.name)
print("Class names:", class_names)
print("No test recordings or files loaded.")
display(val_meta["label"].value_counts().rename("validation clips").to_frame())

## 3 · Check the AIS distance column and convert it to metres

`mean_distance` is a recording-level average, **not a per-second trajectory**. The original selection radius is about 2 km. The code checks the actual numeric distribution before deciding whether the CSV stores kilometres or metres.

In [ ]:
ship_mask = labels_val != "background"
raw_distance = pd.to_numeric(val_meta["mean_distance"], errors="coerce").to_numpy(dtype=float)
usable = raw_distance[ship_mask & np.isfinite(raw_distance)]
if len(usable) == 0:
    raise ValueError("No usable vessel-to-hydrophone distances in validation metadata.")
p50, p95 = np.percentile(usable, [50, 95])
print(f"Raw mean_distance: median={p50:.3f}; 95th percentile={p95:.3f}; max={usable.max():.3f}")
if DISTANCE_UNITS == "auto":
    if 0 < p95 <= 3.0:
        resolved_units = "km"
    elif 10 <= p95 <= 3000:
        resolved_units = "m"
    else:
        raise ValueError("Cannot safely infer mean_distance units. Inspect the values and set DISTANCE_UNITS to 'km' or 'm'.")
else:
    resolved_units = DISTANCE_UNITS
if resolved_units not in {"km", "m"}:
    raise ValueError("DISTANCE_UNITS must be 'auto', 'km', or 'm'")
distance_m = raw_distance * (1000.0 if resolved_units == "km" else 1.0)
print(f"Using {resolved_units} -> metres: ship distance range {np.nanmin(distance_m[ship_mask]):.0f}–{np.nanmax(distance_m[ship_mask]):.0f} m")
print("Missing vessel distances:", int(np.sum(ship_mask & ~np.isfinite(distance_m))))

# Available clips per class in the first example window.
def distance_counts(lower_m, upper_m):
    sel = ship_mask & np.isfinite(distance_m) & (distance_m >= lower_m) & (distance_m <= upper_m)
    return pd.Series(labels_val[sel]).value_counts().reindex(ship_names, fill_value=0).rename("clips")
display(distance_counts(WINDOW_MIN_M, WINDOW_MAX_M).to_frame())

## 4 · Fit a **common, unlabeled** PCA representation

Fit the `StandardScaler` and `PCA` on a reproducible subset of **training features** (labels ignored), then transform **validation features**. The *same PCA axes* are used for all distance windows. K-Means uses 32 PCA dimensions; only the plots use the first two.

In [ ]:
rng = np.random.default_rng(SEED)
fit_rows = rng.choice(len(X_train), size=min(PCA_FIT_SAMPLES, len(X_train)), replace=False)
fit_X = np.asarray(X_train[fit_rows], dtype=np.float32)
if not np.isfinite(fit_X).all():
    raise ValueError("Non-finite values in training feature cache")
scaler = StandardScaler()
fit_scaled = scaler.fit_transform(fit_X)
n_components = min(PCA_COMPONENTS, fit_scaled.shape[0]-1, fit_scaled.shape[1])
pca = PCA(n_components=n_components, svd_solver="randomized", random_state=SEED)
pca.fit(fit_scaled)
del fit_X, fit_scaled

# Using the original 1,206-D validation cache: no FFT/audio recomputation.
val_scaled = scaler.transform(np.asarray(X_val, dtype=np.float32))
Z_val = pca.transform(val_scaled).astype(np.float32)
del val_scaled
print(f"K-Means input: {Z_val.shape[1]} PCA dimensions")
print(f"Variance retained: {pca.explained_variance_ratio_.sum():.1%} (not automatically 95%)")
print("Validation embeddings:", Z_val.shape)

# Fix the display axes, so the two colourings within each experiment are comparable.
PLOT_LIMITS = []
for c in range(2):
    low, high = np.quantile(Z_val[:, c], [0.005, 0.995])
    margin = max((high-low)*0.05, 1e-5)
    PLOT_LIMITS.append((low-margin, high+margin))

## 5 · K-Means, paired comparison plots, and cluster–class heatmap

Cluster IDs (`0`, `1`, etc.) are arbitrary, **not automatically cargo/tug/etc.** The second plot shows the true vessel labels on exactly the same PCA coordinates. ARI and NMI compare assignments with true labels *after* clustering.

In [ ]:
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

PALETTE = list(plt.get_cmap("tab10").colors)
LABEL_COLOURS = {name: PALETTE[i] for i, name in enumerate(class_names)}

def score_kmeans(indices, k, seed=SEED):
    indices = np.asarray(indices, dtype=int)
    if len(indices) < max(30, k * 8):
        raise ValueError(f"Only {len(indices)} clips; too few for this distance window")
    model = KMeans(n_clusters=k, n_init=10, random_state=seed)
    pred = model.fit_predict(Z_val[indices])
    true = labels_val[indices]
    scores = {"n":len(indices), "ari":adjusted_rand_score(true, pred),
              "nmi":normalized_mutual_info_score(true, pred)}
    return pred, scores

def plot_clustering(name, indices, k, save=True):
    indices = np.asarray(indices, dtype=int)
    true = labels_val[indices]
    pred, scores = score_kmeans(indices, k)
    print(f"{name}: {scores['n']:,} clips | K={k} | ARI={scores['ari']:.4f} | NMI={scores['nmi']:.4f}")
    display(pd.Series(true).value_counts().rename("clips").to_frame())

    # Plot a fixed random subset when the selection contains many samples.
    draw_rng = np.random.default_rng(SEED)
    draw = (np.sort(draw_rng.choice(len(indices), PLOT_MAX_POINTS, replace=False))
            if len(indices) > PLOT_MAX_POINTS else np.arange(len(indices)))
    xy = Z_val[indices[draw], :2]
    options = [("K-Means clusters", pred[draw], {i:PALETTE[i % len(PALETTE)] for i in range(k)}),
               ("Actual vessel classes", true[draw], LABEL_COLOURS)]
    for suffix, (heading, colour_values, cmap) in enumerate(options):
        fig, ax = plt.subplots(figsize=(7, 5.5))
        for group, color in cmap.items():
            take = colour_values == group
            if take.any():
                ax.scatter(xy[take,0], xy[take,1], s=7, alpha=0.42,
                           c=[color], label=str(group), linewidths=0, rasterized=True)
        ax.set(xlabel="PCA component 1", ylabel="PCA component 2", title=f"{name}: {heading}")
        ax.set_xlim(*PLOT_LIMITS[0]); ax.set_ylim(*PLOT_LIMITS[1])
        ax.legend(title="Cluster" if suffix==0 else "True class", markerscale=2.0,
                  fontsize=9, loc="best")
        fig.tight_layout()
        if save:
            out = OUT / f"{name}_{'clusters' if suffix==0 else 'actual_classes'}.png"
            fig.savefig(out, dpi=170)
        plt.show(); plt.close(fig)

    counts = pd.crosstab(pd.Series(pred, name="Cluster"), pd.Series(true, name="True class"))
    percent = 100 * counts.div(counts.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(max(6, len(counts.columns)*1.25), max(4, k*.7+1)))
    im = ax.imshow(percent.values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(len(counts.columns)), labels=counts.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(counts.index)), labels=counts.index)
    ax.set(xlabel="Actual vessel class", ylabel="K-Means cluster", title=f"{name}: actual composition of each cluster")
    for row in range(counts.shape[0]):
        for col in range(counts.shape[1]):
            pc = percent.iloc[row,col]; ct = counts.iloc[row,col]
            ax.text(col, row, f"{pc:.0f}%\n({ct})", ha="center", va="center",
                    fontsize=9, color="white" if pc>55 else "black")
    fig.colorbar(im, ax=ax, label="% within cluster")
    fig.tight_layout()
    if save:
        fig.savefig(OUT/f"{name}_cluster_class_heatmap.png", dpi=170)
        counts.to_csv(OUT/f"{name}_cluster_class_counts.csv")
    plt.show(); plt.close(fig)
    return scores

all_results = []

## 6 · Experiment A: **all five classes**, including background

Uses K=5 and all validation distances. This is the direct five-class unsupervised baseline.

In [ ]:
idx_all = np.arange(len(val_meta))
scores = plot_clustering("all_5_classes", idx_all, k=5)
all_results.append({"experiment":"all_5_classes", "distance":"all", "k":5, **scores})

## 7 · Experiment B: **four vessel classes**, all distances

Exclude background because there is no ship–hydrophone distance for it. This is the proper comparator for a distance-filtered four-vessel-class experiment.

In [ ]:
idx_ships = np.flatnonzero(ship_mask & np.isfinite(distance_m))
scores = plot_clustering("vessels_all_distances", idx_ships, k=4)
all_results.append({"experiment":"vessels_all_distances", "distance":"all available vessel distances", "k":4, **scores})

## 8 · Experiment C: **four vessel classes at 400–700 m**

Change the two values in Cell 1 to try another range. The **scaler/PCA does not change**. K-Means is fitted again using only the selected recordings, still without their labels.

In [ ]:
def run_distance_window(lower_m, upper_m, *, save=True):
    if not (0 <= lower_m < upper_m <= 2000):
        raise ValueError("Choose 0 <= lower < upper <= 2000 metres")
    selected = np.flatnonzero(ship_mask & np.isfinite(distance_m) &
                              (distance_m >= lower_m) & (distance_m <= upper_m))
    counts = pd.Series(labels_val[selected]).value_counts().reindex(ship_names, fill_value=0)
    if (counts < 20).any():
        print("CAUTION: fewer than 20 recordings for one or more vessel classes:")
        display(counts.to_frame("clips"))
    name = f"vessels_{lower_m:d}_{upper_m:d}m"
    scores = plot_clustering(name, selected, k=4, save=save)
    return {"experiment":name, "distance":f"{lower_m}–{upper_m} m", "k":4, **scores}

scores = run_distance_window(WINDOW_MIN_M, WINDOW_MAX_M)
all_results.append(scores)
display(pd.DataFrame(all_results))
pd.DataFrame(all_results).to_csv(OUT/"initial_clustering_scores.csv", index=False)

## 9 · Fairer comparison: distance window versus **other distances**, matched by class and sample count

An ARI improvement from the previous cells might simply result from a smaller sample or different class proportions. To check that, this cell draws **the same number of clips from each vessel class** inside and outside the 400–700 m window, then compares K-Means ARI/NMI across 5 repeat samples. Actual labels are used **only to control sample composition and evaluate results**, not as K-Means input.

This remains exploratory: hydrophone, speed, vessel identity and signal-to-noise ratio can still differ between groups.

In [ ]:
def matched_distance_comparison(lower_m=400, upper_m=700, repeats=5, max_per_class=700):
    inside = ship_mask & np.isfinite(distance_m) & (distance_m>=lower_m) & (distance_m<=upper_m)
    outside = ship_mask & np.isfinite(distance_m) & ~inside
    # Equal number of examples of each vessel class in the two compared groups.
    per_class = {}
    for cls in ship_names:
        a = int(np.sum(inside & (labels_val==cls)))
        b = int(np.sum(outside & (labels_val==cls)))
        per_class[cls] = min(a, b, max_per_class)
    display(pd.DataFrame({"matched clips per class":per_class}).T)
    if min(per_class.values()) < 20:
        print("Too few clips of at least one class to draw a meaningful matched comparison.")
        return pd.DataFrame()
    records = []
    for run in range(repeats):
        rg = np.random.default_rng(SEED + run)
        for name, selector in (("within_window",inside), ("outside_window",outside)):
            chunks=[]
            for cls, n in per_class.items():
                candidates = np.flatnonzero(selector & (labels_val == cls))
                chunks.append(rg.choice(candidates, size=n, replace=False))
            indices = np.concatenate(chunks)
            _, metrics = score_kmeans(indices, k=4, seed=SEED+run)
            records.append({"repeat":run+1, "group":name, "n":metrics["n"],
                            "ARI":metrics["ari"], "NMI":metrics["nmi"]})
    results = pd.DataFrame(records)
    display(results.groupby("group")[["ARI","NMI"]].agg(["mean","std"]).round(4))
    out = OUT/f"matched_{lower_m}_{upper_m}m_results.csv"
    results.to_csv(out,index=False)
    print("Saved:", out)
    return results

matched_results = matched_distance_comparison(WINDOW_MIN_M, WINDOW_MAX_M)

## 10 · Optional distance slider (only if `ipywidgets` is installed)

Slide the range, click **Run selected window**. This triggers new K-Means clustering in the **same PCA representation**, not new STFT feature extraction. Notebook outputs for these interactive experiments are displayed but not saved automatically.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
except ImportError:
    print("Optional slider unavailable. Run `%pip install ipywidgets` in a notebook cell, then rerun this cell.")
    print("You can also call run_distance_window(700, 1000) manually.")
else:
    slider = widgets.IntRangeSlider(value=(400,700), min=0, max=2000, step=50,
                                    description="Distance (m)", continuous_update=False,
                                    layout=widgets.Layout(width="650px"))
    run_button = widgets.Button(description="Run selected window", button_style="primary")
    output = widgets.Output()
    def on_run_clicked(_):
        with output:
            clear_output(wait=True)
            a,b = slider.value
            try:
                run_distance_window(int(a), int(b), save=False)
            except ValueError as err:
                print("Cannot cluster this selection:", err)
    run_button.on_click(on_run_clicked)
    display(slider, run_button, output)

## How to interpret the results

- **ARI/NMI** measure agreement between discovered clusters and known classes. A value near zero suggests little agreement; large positive scores indicate stronger class alignment. K-Means still never sees the class labels during fitting.
- **PCA 2D pictures are only a projection.** K-Means uses 32 PCA dimensions, so visible 2D overlap does not necessarily mean full-space overlap.
- **Distance control is a hypothesis, not an automatic improvement.** Compare the matched groups in Cell 9 before concluding that acoustic signatures separate better at similar distances.
- **Adjacent 5-second clips and repeated MMSIs are correlated.** Large or pure clusters might reflect individual vessels, speed, hydrophone deployments, or noise rather than vessel type. Follow-up: inspect MMSI/station compositions and vary distance windows.
- This is **exploratory clustering on the author-provided validation partition**, not a held-out supervised classification score. No test-set features are accessed.

**Outputs:** small `.png` figures and `.csv` results in `benchmarks/results/uniqueship_stft/fold1_unsupervised_clustering/`.